> #  Get Started with Primitives

References:

https://quantum.cloud.ibm.com/docs/en/api/qiskit/primitives

https://quantum.cloud.ibm.com/docs/en/guides/primitive-input-output


<h3>

- primitives are computational building blocks 

- Used in  larger applications 

- whose input units, called **primitive unified blocs (PUBs)**, require quantum resources to efficiently produce outputs for.

- a data structure known as a Primitive Unified Bloc (PUB) to efficiently define vectorized workloads.

-These PUBs are the fundamental unit of work for workload execution. 

- They are used as inputs to the run() method for the Sampler and Estimator primitives, which execute the defined workload as a job

- two most common tasks for quantum computers 
1.  sampling quantum states and 
2. calculating expectation values. 

These tasks motivated the design of the <u>Qiskit primitives: Estimator and Sampler</u>

**Estimator** computes expectation values of observables with respect to states prepared by quantum circuits.

**Sampler** samples the output register from quantum circuit execution.

</h3>

---
> # PUBs are the basic input unit

---

<h4>

A Primitive Unified Bloc (PUB) is a tuple describing one circuit workload. 

The run() method receives a list of PUBs:

```
job = primitive.run([pub1, pub2, ...])

```

Each PUB can contain circuits, parameters, observables, and execution settings depending on the primitive.
</h4>

> # Estimator inputs
<h4>
An Estimator PUB can contain up to four components:

```

(circuit, observables, parameter_values, precision)

```

**Circuit** — a QuantumCircuit, potentially parameterized.

**Observables** — operators such as Pauli, SparsePauliOp, PauliList, or strings.

**Parameter values** — values used to bind circuit parameters.

Precision — optional requested precision for the expectation-value estimates.

```
Circuit + Observable + Parameters
              ↓
        Estimator
              ↓
     Expectation values

```
Example:

```
pub = (circuit, observables, params)
job = estimator.run([pub])
result = job.result()

```

- The circuit can contain Parameters, and parameter values can be supplied as arrays. 

- Observables can be Pauli, SparsePauliOp, PauliList, strings, etc. 


**NOTE:**

A useful feature is **broadcasting:** parameter arrays and observable arrays can be combined using NumPy-style broadcasting, allowing many circuit/observable/parameter combinations to be evaluated efficiently in one PUB.
</h4>

> ## Sampler inputs
<h4>

A Sampler PUB can contain up to three components:

```
(circuit, parameter_values, shots)
```

The circuit **must contain measurement instructions** for the qubits whose output is being sampled. 

parameter_values are optional when the circuit has no runtime parameters, and shots is optional

```
Circuit + Parameters + Shots
              ↓
           Sampler
              ↓
       Measured samples
```

Example:

```
pub = (circuit, params, 1024)
job = sampler.run([pub])
result = job.result()

```

The output is based on **BitArray objects** rather than the older count-dictionary-only model.

</h4>

---
> # Outputs : After execution:

---
<h4>

```
result = job.result()
```

the result is a PrimitiveResult, containing one PubResult for each submitted PUB. 

Each PubResult has:

```
PubResult
├── data
└── metadata

```

data :  the actual computational results

metadata :  implementation/job information when provided 

</h4>



> ## Estimator output

<h4>

Estimator results are stored in the PUB's data container, commonly including:

```
result[0].data.evs
result[0].data.stds

```

- evs — estimated expectation values.

- stds — corresponding standard deviations.

Their shapes follow the broadcasted shape of the PUB inputs. 

</h4>




> ## Sampler output

<h4>
Sampler results are represented using BitArray objects, with one BitArray associated with each classical register in the circuit.


The BitArray represents the sampled measurement data across shots, rather than simply returning the old-style dictionary of counts. 

```
data = result[0].data
bits = data.meas
```

You can convert the samples to counts with:

```
counts = data.meas.get_counts()
```

</h4>

> ## Broadcasting is important,  NumPy-style broadcasting

<h4>

For example:

```
parameters:  (1, 6)
observables: (4, 1)
                    ↓
result:      (4, 6)

```

This means you can evaluate 4 observables × 6 parameter settings = 24 combinations without manually creating 24 PUBs.
</h4>

> ## BitArray :  Sampler's efficient representation of shot data
<h4>

It stores the measured bitstrings in an array-oriented format. 

Intended to be more efficient for processing than immediately converting everything to Python dictionaries. 

It supports operations such as:

```
data.meas.get_counts()
data.beta.slice_bits(...)
data.beta.slice_shots(...)

```

</h4>

> # Note:

<h4>

- Estimator returns one expectation value estimate for each element of the broadcasted shape.

- Parameter value sets are represented by n x m arrays,

- observable arrays are represented by one or more single-column arrays.

</h4>

<h4>

> Parameter value sets are combined with their observable array to create the resulting expectation value estimates

**Example 1: (broadcast single observable)** has a parameter value set that is a 5x1 array and a 1x1 observables array.

----------------> output: 5x1 array

**Example 2: (zip)** has a 5x1 parameter value set and a 5x1 observables array. 

----------------> output: 5x1 array

**Example 3: (outer/product)** has a 1x6 parameter value set and a 4x1 observables array.

----------------> output: 4x6 array

**Example 4: (Standard nd generalization)** has a 3x6 parameter value set array and two 3x1 observables array. 

----------------> output: 3x6 array

</h4>

> #  Implementations of Qiskit Primitives
<h4>

Qiskit provides several implementations of the Estimator and Sampler primitive interfaces. They differ mainly in where computation happens and what resource they use.

1. **EstimatorV2 & SamplerV2**

- IBM Quantum's cloud-based implementations.
- Used to run circuits on IBM Quantum hardware.
- Provide advanced capabilities such as error mitigation.
- Best suited for real quantum-device execution.

2. **StatevectorEstimator & StatevectorSampler**

- Reference/simulator implementations included with Qiskit.
- Run locally using Qiskit's built-in simulation capabilities.
- Based on the qiskit.quantum_info module.
- Produce results from ideal statevector simulations, without hardware noise.
- Useful for testing, development, and understanding primitive behavior.

3. **BackendEstimatorV2 & BackendSamplerV2**

- Wrapper implementations that turn an existing Qiskit backend into a primitive interface.
- Useful when a quantum-computing provider does not directly offer a primitives-based API.
- Work similarly to the regular Estimator and Sampler.
- Require a backend argument to specify the quantum resource to use.

</h4>


![](./pic1.png)

![](pic2.png)